# Grouped splits: holdout + 5-fold CV

Rebuilds the train/val/test division from `04_split_and_augment` with two changes, and writes it to `data/splits_grouped/`. Notebook 04 and `data/splits/` are left exactly as they are — this sits alongside them.

**Why redo the split at all.** Notebook 04 splits 70/15/15 stratified by composer but random by *file*. Multi-movement works therefore scatter across train, val and test: `Symphony_n39_K543_1mov` can be in train while `..._4mov` is in test. Those are the same piece, usually the same transcriber and the same performance conventions, so the model can recognise the sibling rather than the composer. Measured below — it affects roughly one in seven val and test rows.

**Why cross-validation on top.** Chopin has 20 files in the current test set. One flipped prediction moves chopin recall by five points, which is wider than any plausible difference between the two models. A single split cannot support the sentence "the CNN reads style better than the LSTM"; five folds with a standard deviation can.

Outputs:

- `data/splits_grouped/holdout.csv` — ~14% of works, evaluated **once** per model at the very end
- `data/splits_grouped/folds.csv` — the rest, tagged `fold` 0–4 for cross-validation

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
np.random.seed(SEED)

PROC_DIR = Path("../data/processed")
SPLIT_DIR = Path("../data/splits")
OUT_DIR = Path("../data/splits_grouped")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COMPOSERS = ["bach", "beethoven", "chopin", "mozart"]

# Source of truth is the intersection of the two processed manifests, not
# data/raw/manifest.csv. Two of the 1637 files failed to parse in notebooks 02 and
# 03 (corrupt key signature metadata) and have no .npz/.npy on disk, so a split
# built from the raw manifest would reference files that do not exist. Notebook 04
# intersected the same way.
lstm_man = pd.read_csv(PROC_DIR / "lstm_manifest.csv")
cnn_man = pd.read_csv(PROC_DIR / "cnn_manifest.csv")
manifest = lstm_man[["composer", "filename"]].merge(
    cnn_man[["composer", "filename"]], on=["composer", "filename"])

print(f"{len(lstm_man)} lstm / {len(cnn_man)} cnn -> {len(manifest)} usable by both")
manifest.head()

1635 lstm / 1635 cnn -> 1635 usable by both


,composer,filename
0,bach,AveMaria.mid
1,bach,01_Menuet.mid
2,bach,02_Menuet.mid
3,bach,03_Menuet.mid
4,bach,04_Menuet.mid


## Work key

The grouping key strips two things from the filename stem: the augmentation tag (`_shiftp2`) and explicit movement markers (`_4mov`, `_mvt2`, `_part3`, `_satz1`).

It deliberately does **not** strip trailing work numbers like `_n1` or `_No6`. `bwv1066_orchestral_suite_n1` and `bwv1068_orchestral_suite_n3` are genuinely different works and must stay in different groups. That makes the key conservative: it will merge everything it is sure about and leave some real sibling relationships undetected, so the leakage figure below is a **lower bound**, not an exact count.

In [2]:
MOVEMENT = re.compile(r"[_-]?\d*(mov|mvt|movement|part|satz)[_.]?\d*$", re.I)
AUGMENT = re.compile(r"_shift[pm]\d+$")


def work_key(filename):
    """Filename -> a key shared by all movements of the same work."""
    s = AUGMENT.sub("", Path(filename).stem)
    s = MOVEMENT.sub("", s)
    return re.sub(r"[_\W]+$", "", s).lower()


files = manifest.copy()
files["work"] = files["filename"].map(work_key)

print(f"{len(files)} files -> {files['work'].nunique()} works")
multi = files.groupby("work").size().sort_values(ascending=False)
print(f"works with more than one file: {(multi > 1).sum()}")
multi.head(10)

1635 files -> 1445 works
works with more than one file: 77


work
bwv0806_english_suite_n1       10
bwv0811_english_suite_n6        8
bwv1067_orchestral_suite_n2     7
bwv1066_orchestral_suite_n1     7
k317_coronation_mass            6
bwv1069_orchestral_suite_n4     5
bwv1068_orchestral_suite_n3     5
bwv0997_partita_for_lute        5
string_quartet_n2_op18_n2       4
viennese_sonatinas_k439b_n2     4
dtype: int64

In [3]:
# How bad is the leakage in the existing split? This is the number that justifies
# the whole notebook, so it goes in the report.
old = {s: pd.read_csv(SPLIT_DIR / f"{s}.csv") for s in ["train", "val", "test"]}
for s, df in old.items():
    col = "source_filename" if "source_filename" in df.columns else "filename"
    df["work"] = df[col].map(work_key)

train_works = set(old["train"]["work"])
for s in ["val", "test"]:
    hit = old[s]["work"].isin(train_works)
    print(f"{s}: {hit.sum()}/{len(old[s])} rows ({hit.mean():.1%}) share a work with train")
    print("   e.g.", list(old[s].loc[hit, 'work'].unique()[:4]))

val: 39/245 rows (15.9%) share a work with train
   e.g. ['piano_concerto_n2', 'symphony_n33_k319', 'bwv1066_orchestral_suite_n1', 'piano_concerto_n12_k414']
test: 35/246 rows (14.2%) share a work with train
   e.g. ['bwv1068_orchestral_suite_n3', 'symphony_n1', 'bwv0811_english_suite_n6', 'symphony_n7']


## Holdout, then folds

`StratifiedGroupKFold` keeps every work entirely inside one fold while holding the composer mix roughly constant. Taking 1 of 7 folds as the holdout gives ~14%, close to the 15% notebook 04 used; the remaining 6/7 is then re-split into 5 CV folds.

This is a split of **original files only**. Augmentation is applied per fold at load time in notebook 08, never written to disk, so an augmented copy can never appear on the other side of a fold boundary from its original.

In [4]:
y = files["composer"].to_numpy()
groups = files["work"].to_numpy()

# 1 of 7 -> holdout (~14%)
splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
rest_idx, hold_idx = next(splitter.split(files, y, groups))
holdout = files.iloc[hold_idx].reset_index(drop=True)
rest = files.iloc[rest_idx].reset_index(drop=True)

# the remainder -> 5 CV folds
rest["fold"] = -1
folder = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for k, (_, val_idx) in enumerate(folder.split(rest, rest["composer"], rest["work"])):
    rest.loc[val_idx, "fold"] = k
assert (rest["fold"] >= 0).all()

holdout.to_csv(OUT_DIR / "holdout.csv", index=False)
rest.to_csv(OUT_DIR / "folds.csv", index=False)

print(f"holdout {len(holdout)} files / {holdout['work'].nunique()} works")
print(f"folds   {len(rest)} files / {rest['work'].nunique()} works")

holdout 234 files / 206 works
folds   1401 files / 1239 works


In [5]:
# The checks that matter: no work may appear on both sides of any boundary, and
# every referenced file must actually exist in both processed directories.
hold_works = set(holdout["work"])
assert not (hold_works & set(rest["work"])), "work leaked between holdout and folds"

for k in range(5):
    va = set(rest.loc[rest["fold"] == k, "work"])
    tr = set(rest.loc[rest["fold"] != k, "work"])
    assert not (va & tr), f"work leaked across fold {k}"

for df in (holdout, rest):
    for r in df.itertuples():
        stem = Path(r.filename).stem
        assert (PROC_DIR / "lstm" / r.composer / f"{stem}.npz").exists(), stem
        assert (PROC_DIR / "cnn" / r.composer / f"{stem}.npy").exists(), stem

print("no work appears on both sides of any split boundary")
print("every referenced file exists in both processed directories")
print("\nfiles per composer per fold:")
pd.crosstab(rest["fold"], rest["composer"])

no work appears on both sides of any split boundary
every referenced file exists in both processed directories

files per composer per fold:


composer,bach,beethoven,chopin,mozart
fold,,,,
0,176,37,24,44
1,175,37,24,44
2,175,38,23,44
3,176,38,23,43
4,176,37,23,44


In [6]:
print("holdout composer mix:")
print(holdout["composer"].value_counts())
print("\nsmallest class in the holdout:", holdout["composer"].value_counts().min(), "files")
print("(the single-split version had 20 chopin files in test — this is why we also cross-validate)")

holdout composer mix:
composer
bach         146
mozart        37
beethoven     32
chopin        19
Name: count, dtype: int64

smallest class in the holdout: 19 files
(the single-split version had 20 chopin files in test — this is why we also cross-validate)


## Notes

- `data/splits/` (notebook 04) is untouched. Notebooks 05–07 still read it, so the original single-split results stay reproducible for comparison in the report.
- Built from the processed manifests, not the raw one, so the two unparseable MIDI files are excluded: 1635 files, not 1637.
- The work key is conservative by design — the leakage it reports for the old split is a floor.
- Grouping is by work, not by source folder. Folder-level grouping would be stricter still, but it would put all 400 Bach chorales in one group and make the folds badly lumpy.
- Holdout is written here and must not be read again until the final evaluation cell in notebook 08.